In [1]:
import pandas as pd
import numpy as np
import os
from functools import reduce

print("Iniciando Script 3: Feature Engineering para el Modelo de Ozono...")

# 1. CARGAMOS SOLO LAS VARIABLES NECESARIAS
# Elegimos las variables clave discutidas para la "cocina" del Ozono y el clima
variables = ['O3', 'NOX', 'SR', 'TOUT', 'WSR', 'WDR']
dfs = []

for var in variables:
    ruta = f"data/processed/variables/{var}_clean.parquet"
    if os.path.exists(ruta):
        df_var = pd.read_parquet(ruta)
        dfs.append(df_var)
    else:
        print(f"⚠️ Advertencia: No se encontró {ruta}")

# 2. UNIMOS LAS TABLAS (Inner Join)
# Solo conservaremos las horas donde TODAS estas variables estén funcionando bien
print("Cruzando variables limpias en una sola matriz...")
df_ml = reduce(lambda left, right: pd.merge(left, right, on=['Date', 'Estacion'], how='inner'), dfs)

# Ordenamos estrictamente por tiempo para que los lags funcionen bien
df_ml.sort_values(by=['Estacion', 'Date'], inplace=True)
df_ml.reset_index(drop=True, inplace=True)

# 3. TRANSFORMACIÓN VECTORIAL DEL VIENTO (U, V)
print("Convirtiendo grados circulares a vectores cartesianos (U y V)...")
# Convertimos grados a radianes y calculamos componentes
radianes = df_ml['WDR'] * (np.pi / 180)
df_ml['U_Wind'] = -df_ml['WSR'] * np.sin(radianes)
df_ml['V_Wind'] = -df_ml['WSR'] * np.cos(radianes)

# Ya podemos tirar los grados circulares originales para que no confundan al modelo
df_ml.drop(columns=['WDR', 'WSR'], inplace=True)

# 4. CREACIÓN DE TIME-LAGS (La "Memoria" del Modelo)
print("Generando rezagos temporales (Time-Lags)...")
# Agrupamos por estación para no mezclar el pasado de 'Centro' con el presente de 'Santa Catarina'
estaciones = df_ml.groupby('Estacion')

# Lags de 1 a 4 horas para la formación de gases y radiación
for lag in [1, 2, 3, 4]:
    df_ml[f'NOX_lag_{lag}'] = estaciones['NOX'].shift(lag)
    df_ml[f'SR_lag_{lag}'] = estaciones['SR'].shift(lag)

# Lags de 2 y 4 horas para temperatura (Inversión térmica)
for lag in [2, 4]:
    df_ml[f'TOUT_lag_{lag}'] = estaciones['TOUT'].shift(lag)

# Lags de 1 y 2 horas para el viento (Transporte)
for lag in [1, 2]:
    df_ml[f'U_Wind_lag_{lag}'] = estaciones['U_Wind'].shift(lag)
    df_ml[f'V_Wind_lag_{lag}'] = estaciones['V_Wind'].shift(lag)

# 5. LIMPIEZA FINAL Y EXPORTACIÓN
# Al desfasar datos, las primeras filas de cada estación se vuelven NaN (porque no tienen pasado)
df_ml_final = df_ml.dropna().reset_index(drop=True)

os.makedirs("data/ml_ready", exist_ok=True)
ruta_final = "data/ml_ready/dataset_ozono_predictivo.parquet"
df_ml_final.to_parquet(ruta_final, index=False)

print(f"\n✅ ¡Transformación Matemática Lista!")
print(f"Dataset predictivo guardado en: {ruta_final}")
print(f"Filas listas para Machine Learning: {df_ml_final.shape[0]:,}")
print("Nuevas columnas generadas:")
print(list(df_ml_final.columns))

Iniciando Script 3: Feature Engineering para el Modelo de Ozono...
Cruzando variables limpias en una sola matriz...
Convirtiendo grados circulares a vectores cartesianos (U y V)...
Generando rezagos temporales (Time-Lags)...

✅ ¡Transformación Matemática Lista!
Dataset predictivo guardado en: data/ml_ready/dataset_ozono_predictivo.parquet
Filas listas para Machine Learning: 536,075
Nuevas columnas generadas:
['Date', 'Estacion', 'O3', 'NOX', 'SR', 'TOUT', 'U_Wind', 'V_Wind', 'NOX_lag_1', 'SR_lag_1', 'NOX_lag_2', 'SR_lag_2', 'NOX_lag_3', 'SR_lag_3', 'NOX_lag_4', 'SR_lag_4', 'TOUT_lag_2', 'TOUT_lag_4', 'U_Wind_lag_1', 'V_Wind_lag_1', 'U_Wind_lag_2', 'V_Wind_lag_2']
